# Level 3 · Day & Night: sunlight, NO₃ and N₂O₅ uptake

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reflective-org/AAAR-2026-Aerosol-Tutorial/blob/main/notebooks/03_day_and_night.ipynb)

In Levels 1 and 2 the light was off or held constant. Here the box sits at a real place on a
real date. Starting at local noon in Buffalo, NY on 1 July, TUV-x computes the photolysis rates
from the position of the sun over 24 hours. By day, sunlight makes OH and destroys NO₃ within
seconds. After sunset NO₃ and N₂O₅ build up, and N₂O₅ that reacts on the particles (uptake
coefficient γ) becomes particle nitrate. In this notebook you compute the sunlight for different
sites and dates, run the full model (MCM limonene chemistry coupled to TOMAS aerosol
microphysics), and change γ, the season, the site and the NOx source to see what controls
nighttime nitrate. The Level 3 tab of the tutorial web app runs the same model with sliders;
this notebook shows what is under the hood.

## Learning goals

After this notebook you should be able to:

1. Explain how latitude, longitude, date and time set the solar zenith angle, and how that sets
   the photolysis frequencies $J$.
2. Describe the day–night cycle of OH and NO₃, and why N₂O₅ builds up only at night.
3. Write down the N₂O₅ uptake rate $k_{het} = \gamma\,\bar c\,A/4$ and estimate it from a size
   distribution.
4. Explain why particle nitrate grows less than proportionally with γ.
5. Explain what "holding NO₂ constant" does to the nitrogen budget of a box model.

**How long things take.** On Colab the first install takes 1–2 minutes. A 24-hour Level 3 run
takes about 20–30 s on a laptop, and up to about twice that on Colab. The first run in a session
also compiles the model, which adds about 5 s once (more on Colab). Run top to bottom, the notebook makes
7 model runs: about 2–4 minutes on a laptop. To skip the first run, set `USE_PRECOMPUTED = True`
in section 2; that loads a saved copy of the preset run, so its plots appear at once.

## Setup

Run the next two cells once per session.

- **Colab:** the first cell installs the box model and the tutorial helpers (1–2 minutes). Red pip
  "dependency conflicts" lines are expected. If a cell asks you to restart, click
  *Runtime › Restart session*, then *Runtime › Run all*.
- **Your own computer:** install the repository first ([README](https://github.com/reflective-org/AAAR-2026-Aerosol-Tutorial/blob/main/README.md#run-on-your-own-computer));
  the first cell then does nothing.
- `at.setup()` downloads the TUV-x photolysis data once (about 5 MB) and prints the versions.

In [ ]:
# Colab: installs the tutorial helpers and the box model (first time: 1-2 minutes).
# Red pip "dependency conflicts" lines on Colab are expected and do not matter here.
import importlib.metadata
import sys

HELPERS_VERSION = "0.3.0"   # the aaar_tutorial version these notebooks need


def _older(version):
    return tuple(map(int, version.split("."))) < tuple(map(int, HELPERS_VERSION.split(".")))


try:
    installed = importlib.metadata.version("aaar-tutorial")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed is None or _older(installed):
    if "google.colab" not in sys.modules:
        raise ModuleNotFoundError(
            f"This notebook needs the tutorial helpers {HELPERS_VERSION}, found {installed or 'none'}. Pick the "
            "kernel of the repository's .venv, update it (git pull, then uv sync or pip install -e .), then "
            "restart the kernel.")
    %pip install -q "aaar-tutorial @ git+https://github.com/reflective-org/AAAR-2026-Aerosol-Tutorial@main"
    importlib.invalidate_caches()           # so the next cell finds the new packages
loaded = getattr(sys.modules.get("aaar_tutorial"), "__version__", None)
if loaded is not None and _older(loaded):
    raise RuntimeError("This session still has older tutorial helpers loaded. Colab: Runtime > Restart "
                       "session, then Runtime > Run all. Locally: Kernel > Restart Kernel.")

In [ ]:
import aaar_tutorial as at
from aaar_tutorial import config, plots

at.setup()

In [ ]:
#@title Size resolution
# Size bins for every run in this notebook. 40 (the web app's grid) is the fastest; 80, 160 and 320
# draw the size distribution more finely (results change by under 1 %) and take longer: a Level 1
# run takes about 1, 4, 14 and 60 s on a laptop, Level 3 about 12, 19, 39 and 140 s; Colab is
# about twice as slow. Rerun the cells below after changing it.
N_BINS = 40  #@param [40, 80, 160, 320] {type:"raw"}
at.use_bins(N_BINS)

In [ ]:
import html
import inspect
from datetime import date

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
from IPython.display import display

PRESET = "level3_day_night"

## 1. Where is the sun? Solar zenith angle and photolysis rates

The photolysis frequency of a molecule (units s⁻¹) is

$$J = \int \sigma(\lambda, T)\,\phi(\lambda, T)\,F(\lambda)\,d\lambda$$

where $\sigma$ is its absorption cross section, $\phi$ the quantum yield of the channel and $F$
the actinic flux (photons cm⁻² s⁻¹ nm⁻¹). TUV-x computes $F$ for a clear-sky standard atmosphere.
At the surface, $F$ depends mainly on the **solar zenith angle** $\chi$ (SZA), the angle between
the sun and the vertical:

$$\cos\chi = \sin\varphi\,\sin\delta + \cos\varphi\,\cos\delta\,\cos h$$

with latitude $\varphi$, solar declination $\delta$ (from −23.4° in late December to +23.4° in
late June) and hour angle $h$ (15° per hour away from solar noon). When the sun is below the
horizon, $\chi > 90°$ and every $J$ is zero.

In the model, TUV-x computes $J(\chi)$ once per session on a 1° grid; the model interpolates
it and scales it by the Earth–Sun distance. The ozone column, the surface albedo (0.1) and the
clear sky are fixed, and aerosols do not change the light (ASSUMPTIONS A7, A11).

The cells below compute $\chi$ and $J$ for any site and date directly from the photolysis
module, without running the chemistry (a few seconds). `at.site_overrides(site, day)` gives the run
inputs that put the box at one of the web app's sites (listed in `config.SITES`) from local noon,
and `at.sunlight_over_day(site, day)` asks TUV-x for $\chi$ and $J$ over the next 24 hours.

In [ ]:
for site, (lat, lon, tz) in config.SITES.items():
    print(f"{site:20s} {lat:7.2f} °N {lon:8.2f} °E   {tz}")

at.site_overrides("Buffalo, NY", "2026-07-01")   # the preset: 12:00 EDT = 16:00 UTC

In [ ]:
SUN_CASES = [("Buffalo, NY", "2026-07-01"), ("Buffalo, NY", "2026-01-15"),
             ("Hyytiälä, Finland", "2026-07-01"), ("Manaus, Brazil", "2026-07-01")]

days = [at.sunlight_over_day(site, day) for site, day in SUN_CASES]   # 24 h from local noon
print(f"{'site and date':32s} {'daylight (h)':>12s} {'min SZA (°)':>12s} {'max J(NO₂) (s⁻¹)':>17s}")
for d in days:
    step_h = d["time_h"][1] - d["time_h"][0]
    daylight_h = step_h * np.sum(d["sza_deg"][:-1] < 90)
    print(f"{d['label']:32s} {daylight_h:12.2f} {d['sza_deg'].min():12.1f} {d['J_NO2'].max():17.2e}")
plots.sunlight(days)

**Try this:** add cases to `SUN_CASES`, for example `("Hyytiälä, Finland", "2026-12-21")`
(the shortest day) or `("Manaus, Brazil", "2026-12-21")`. To add your own site, add it to
`config.SITES` with its latitude, longitude and [time-zone name](https://en.wikipedia.org/wiki/List_of_tz_database_time_zones),
e.g. `config.SITES["Boulder, CO"] = (40.01, -105.27, "America/Denver")`.

Note that J(NO₂) depends on the time of day only through $\chi$: the curve is highest when the
sun is highest, and the daylight hours are where SZA is below the dashed 90° line.

## 2. Run the Level 3 preset

`at.describe` lists every input of the preset; any of them can be changed in `at.run(...)`.

In [ ]:
at.describe(PRESET)

What the preset sets up:

- **Place and time:** Buffalo, NY (42.89° N, 78.88° W), 1 July, 24 hours from 12:00 local time
  (`start_utc` is 16:00 UTC). Temperature (20 °C) and RH (60 %) stay fixed all day.
- **Gases:** 2 ppb limonene (it decays: it is not held), 40 ppb O₃, 0.5 ppb NO, 2 ppb SO₂, CO and
  CH₄. **NO₂ is held at 1 ppb** (`hold_constant = ("NO2",)`) as a stand-in for continuous NOx
  emissions (section 7).
- **Particles:** 3000 cm⁻³ seed at 80 nm (30 % non-volatile organic, 70 % sulfate).
- **Uptake:** `gamma_n2o5 = 0.02`.
- **No emissions, dilution or deposition** (ASSUMPTIONS A14), so O₃ accumulates over the run.

Every 15 minutes (`coupling_dt_s = 900`) the model: (1) takes $J$ at the middle of the interval
and computes $k_{het}$ for N₂O₅ from the current particles; (2) integrates the MCM limonene
chemistry for 15 minutes with a stiff JAX solver; (3) hands H₂SO₄, condensable organics and the
new nitrate to TOMAS; (4) steps the aerosol (coagulation, condensation, SOA partitioning, water);
(5) resets NO₂ to 1 ppb. Results are saved every 30 minutes.

In [ ]:
USE_PRECOMPUTED = False   # True: load the saved preset run (instant) instead of running the model

if USE_PRECOMPUTED:
    base = at.load_example(PRESET)
else:
    base = at.run(PRESET)       # about 20-30 s; the first run in a session also compiles the model

summary = at.summary(base)
for key in ("N_cm3", "dry_ugm3", "SO4_ugm3", "SOA_ugm3", "NIT_ugm3"):
    print(f"{key:9s} {summary[key + '_start']:8.3g} -> {summary[key + '_end']:8.3g}")
print(f"GMD (nm)  {summary['gmd_nm_start']:8.3g} -> {summary['gmd_nm_end']:8.3g}")
print("Nights (h since start):", [(round(a, 2), round(b, 2)) for a, b in at.night_intervals(base)])

In [ ]:
plots.photolysis(base, j=("J_NO2", "J_O3_O1D", "J_NO3_NO2"))

Shaded bands mark the night (SZA > 90°): here from about 20:53 to 05:46 local time. Note the
size of J(NO₃→NO₂): about 0.2 s⁻¹ at noon, so a NO₃ radical survives only a few seconds in
sunlight. Hover over the curves to read values.

## 3. Day and night chemistry

**By day**, photolysis drives the radicals:

$$\mathrm{O_3} + h\nu \rightarrow \mathrm{O(^1D)} + \mathrm{O_2}, \qquad
\mathrm{O(^1D)} + \mathrm{H_2O} \rightarrow 2\,\mathrm{OH}$$

$$\mathrm{NO_2} + h\nu \rightarrow \mathrm{NO} + \mathrm{O}, \qquad
\mathrm{NO_3} + h\nu \rightarrow \mathrm{NO_2} + \mathrm{O}, \qquad
\mathrm{NO_3} + \mathrm{NO} \rightarrow 2\,\mathrm{NO_2}$$

OH oxidizes limonene, SO₂ (to H₂SO₄) and NO₂ (to HNO₃). NO₃ is made all the time but is
photolysed or reacts with NO within seconds.

**At night**, photolytic OH production stops (OH falls about 100-fold; a little still comes from
ozonolysis), NO is titrated by O₃, and NO₃ survives:

$$\mathrm{NO_2} + \mathrm{O_3} \rightarrow \mathrm{NO_3} + \mathrm{O_2}
\qquad k_1 = 1.4\times10^{-13}\,e^{-2470/T}\ \mathrm{cm^3\,s^{-1}}$$

$$\mathrm{NO_3} + \mathrm{NO_2} + \mathrm{M} \rightleftharpoons \mathrm{N_2O_5} + \mathrm{M}$$

The equilibrium is fast, and colder air shifts it towards N₂O₅. NO₃ and N₂O₅ act as one
reservoir that loses nitrogen to the particles (section 4).

In [ ]:
plots.radicals(base).show()
plots.gases(base).show()

**OH is high here.** Daytime OH reaches about 2×10⁷ molec cm⁻³ (1.7×10⁷ on the first afternoon,
2.3×10⁷ at noon on the second day), several times typical ambient peaks. Holding NO₂ at 1 ppb keeps
HO₂ + NO recycling OH, there is little VOC left to remove OH once the limonene is gone, and O₃
builds up without deposition (ASSUMPTIONS A14). Radicals start at zero and spin up within seconds;
the first saved point is at 30 minutes.

**Try this:** in the gases plot, click `Limonene` and `NO` in the legend to show them. Limonene
is used up within about 2 hours of the sunny afternoon. NO falls to almost zero at night, when
nothing photolyses NO₂ and O₃ removes the NO that is left. O₃ rises from 40 to about 80 ppb:
the box makes O₃ by day and has no deposition to remove it.

**Under the hood:** the reactions above, as they appear in the mechanism file the model reads.
`<34>` is the chamber's fixed N₂O₅ loss (4×10⁻⁴ s⁻¹). The driver overwrites its rate constant
every 15 minutes with $k_{het}$ from the particles and sends its product (`NA_HET`) to the
particles as nitrate. `<33>`, the HNO₃ loss, stays a gas-phase sink: it does not make particle
nitrate (ASSUMPTIONS A1, A2).

In [ ]:
from tomas_jax.box.coupling import tutorial_mechanism

mechanism = tutorial_mechanism()
by_label = {r.label: r for r in mechanism.reactions}
print(f"{len(mechanism.species)} species, {len(mechanism.reactions)} reactions\n")
for label in ("8", "12", "44", "40", "41", "10", "34", "33"):
    r = by_label[label]
    products = " + ".join(p if n == 1 else f"{n:g} {p}" for p, n in r.products)
    print(f"<{label:>2}> {' + '.join(r.reactants):9s} -> {products:12s} k = {r.rate_expr}")

## 4. N₂O₅ uptake makes particle nitrate

N₂O₅ that hits a particle reacts with a probability γ (the reactive uptake coefficient) and
forms two nitrate ions:

$$\mathrm{N_2O_5(g)} + \mathrm{H_2O(p)} \rightarrow 2\,\mathrm{HNO_3} \rightarrow 2\,\mathrm{NO_3^-(p)}$$

For small particles (free-molecular regime) the first-order loss rate of N₂O₅ is

$$k_{het} = \frac{\gamma\,\bar c\,A}{4}, \qquad \bar c = \sqrt{\frac{8RT}{\pi M_{\mathrm{N_2O_5}}}}$$

where $A$ is the particle surface area per volume of air and $\bar c$ the mean molecular speed
(about 240 m s⁻¹ at 20 °C). The nitrate then grows as

$$\frac{d[\mathrm{NO_3^-}]}{dt} = 2\,k_{het}\,[\mathrm{N_2O_5}]$$

The model computes $k_{het}$ like a condensation sink with γ in place of the accommodation
coefficient. It uses a Fuchs–Sutugin correction for gas diffusion to larger particles, and the
**wet** particle diameters. The nitrate goes to each size bin in proportion to its share of
$k_{het}$. Nitrate is non-volatile (no NH₄NO₃ equilibrium) and γ does not change with RH or
composition (ASSUMPTIONS A5; `docs/aaar/heterogeneous.md` in TOMAS-JAX). SOA takes up water like
ammonium bisulfate here (ASSUMPTIONS A9), far more than real SOA, so the wet surface and $k_{het}$
of organic-rich particles are overestimated.

In [ ]:
plots.composition(base).show()
plots.time_series(base, ["N2O5", "NIT_ugm3"]).show()

Nitrate appears only after sunset and grows almost linearly through the night. That is partly how
this model is built: N₂O₅ uptake is its only way to make particle nitrate. The HNO₃ made by day
(OH + NO₂; 3.8 ppb by the end of the run, about 10 µg m⁻³ if it were all particle nitrate) stays in
the gas: there is no ammonia, so no NH₄NO₃ forms, and `<33>` removes HNO₃ as a gas-phase sink
(ASSUMPTIONS A2, A5). In the real atmosphere daytime HNO₃ often condenses as ammonium nitrate.

Sulfate grows by day (SO₂ + OH). SOA forms in the first hours, while limonene is being oxidized,
and peaks near 10 µg m⁻³ about 4 hours in. It then slowly evaporates, mostly the next morning: OH
and thermal decomposition destroy the gas-phase PANs and nitrates, and the particles give some back
to keep up with the gas (an effect of the placeholder c\*, A8).

> **SOA caveat.** SOA here uses the same placeholder volatilities as Level 2 (Li et al. 2016,
> from elemental composition), which make almost all of it organic nitrates and PANs
> (ASSUMPTIONS A8). Treat the SOA amounts as illustrative. This notebook is about the inorganic
> nitrate.

### Under the hood: estimate $k_{het}$ yourself

The next cell computes $A$ from the size distribution in the result, then
$k_{het} = \gamma \bar c A/4$. It compares that with the rate the model actually used over the
night, which is the nitrate formed divided by twice the time-integrated N₂O₅.

In [ ]:
gamma = base.scenario["gamma_n2o5"]
temp_k = base.scenario["environment"]["temp_k"]
c_bar = np.sqrt(8 * config.GAS_CONSTANT * temp_k / (np.pi * config.MOLAR_MASS_G["N2O5"] * 1e-3))   # m/s

# Surface area per volume of air from the reported (dry) size distribution (m² per m³)
d_m = np.asarray(base.diameter_nm) * 1e-9
number_cm3 = np.asarray(base.dndlog10dp_cm3) * np.diff(np.log10(base.diameter_edges_nm))
area = (np.pi * d_m**2 * number_cm3 * 1e6).sum(axis=1)
k_estimate = gamma * c_bar * area / 4

# What the model did over the night: nitrate formed / (2 × time-integrated N2O5), both in ppb
night = at.is_night(base)
t_s = np.array(base.time_h)[night] * 3600
nitrate_ppb = at.series(base, "NIT_ugm3")[night] / at.ugm3_per_ppb("NO3", base)
k_model = (nitrate_ppb[-1] - nitrate_ppb[0]) / (2 * np.trapezoid(at.series(base, "N2O5")[night], t_s))

k_est = k_estimate[night].mean()
print(f"mean speed of N2O5     c̄ = {c_bar:.0f} m/s")
print(f"night-mean dry surface A = {area[night].mean() * 1e6:.0f} µm² cm⁻³")
print(f"estimate (dry A)   k_het = {k_est:.2e} s⁻¹  -> N2O5 lifetime {1 / k_est / 3600:.1f} h")
print(f"model              k_het = {k_model:.2e} s⁻¹  -> N2O5 lifetime {1 / k_model / 3600:.1f} h")
print(f"model / estimate         = {k_model / k_est:.2f}")

The estimate lands within about 30 % of the model's rate, but two effects partly cancel. The
reported diameters assume a density of 1400 kg m⁻³, while the model's dry particles are about
1900 kg m⁻³ (it counts organics and nitrate as sulfate for density, ASSUMPTIONS A9), so this "dry"
area is about 20 % too large. The model, on the other hand, uses the wet surface: at 60 % RH water
(`H2O_ugm3` in the totals) adds about 45 % to it. Because the model gives SOA the water uptake of
ammonium bisulfate (A9), that water, and with it $k_{het}$, is likely too high for these
organic-rich particles. For these particle sizes and γ ≪ 1 the Fuchs–Sutugin correction changes the
rate by only a few percent. [Notebook 4](04_inside_the_model.ipynb), Section 4, computes $k_{het}$
from the model's wet diameters.

Here is where the model computes $k_{het}$ each coupling interval (the lines of `driver.run`
that mention `k_het` or `gamma_n2o5`):

In [ ]:
from tomas_jax.box import driver


def show_source_lines(func, *words):
    """Print the lines of a function's source code that contain any of `words`."""
    for line in inspect.getsource(func).splitlines():
        if any(word in line for word in words):
            print(line.strip())


show_source_lines(driver.run, "k_het", "gamma_n2o5")

## 5. Exercise 1: how much does γ matter?

Laboratory values of γ(N₂O₅) on aqueous sulfate are around 10⁻². Ambient values span roughly
10⁻⁴ to 10⁻¹. Particle nitrate (the "nitrate effect") and organic coatings slow the uptake, and
chloride diverts part of it to ClNO₂, which this model does not include (Bertram & Thornton
2009; Chang et al. 2011).

The cell below runs the preset with each γ in `GAMMAS` and reuses the preset run for γ = 0.02
(3 new runs, about 1 minute).

**Try this:** change `GAMMAS`, for example to `[0.0, 0.005, 0.05, 0.2]`.

In [ ]:
GAMMAS = [0.0, 0.002, 0.02, 0.1]

sweep = {}
for g in GAMMAS:
    if g == base.scenario["gamma_n2o5"]:
        sweep[g] = base                      # already run above
    else:
        sweep[g] = at.run(PRESET, gamma_n2o5=g, name=f"γ = {g:g}")

In [ ]:
gamma_labels = [f"γ = {g:g}" for g in sweep]
plots.time_series(list(sweep.values()), ["N2O5", "NO3", "NIT_ugm3"], labels=gamma_labels)

### Why more γ gives diminishing returns

With NO₂ held constant, every NO₃ made at night can take one NO₂ to form N₂O₅, and every N₂O₅
taken up gives two nitrate ions. Nighttime nitrate therefore cannot exceed twice the NO₃
production:

$$[\mathrm{NO_3^-}]_{max} \approx 2\int_{night} k_1\,[\mathrm{NO_2}]\,[\mathrm{O_3}]\,dt$$

Very small γ (below about 0.002 here): uptake is slow, N₂O₅ piles up, and nitrate grows roughly in
proportion to γ. Already between 0.002 and 0.02, ten times more γ gives only about five times more
nitrate. Large γ:
almost every NO₃ ends up as nitrate, and NO₃ production (NO₂ + O₃), not γ, limits it.

In [ ]:
def nitrate_ceiling_ugm3(result):
    """2 ∫ k1 [NO2][O3] dt over the night, as particle nitrate (µg/m³)."""
    env = result.scenario["environment"]
    k1 = 1.4e-13 * np.exp(-2470 / env["temp_k"])                          # cm³ molec⁻¹ s⁻¹ (MCM <8>)
    air = env["pres_pa"] / (config.BOLTZMANN * env["temp_k"]) * 1e-6      # molecules per cm³
    p_no3_ppb_s = k1 * at.series(result, "NO2") * at.series(result, "O3") * 1e-9 * air
    t_s = np.array(result.time_h) * 3600
    ceiling_ppb = 2 * np.trapezoid(np.where(at.is_night(result), p_no3_ppb_s, 0.0), t_s)
    return ceiling_ppb * at.ugm3_per_ppb("NO3", result)


ceiling = nitrate_ceiling_ugm3(base)
print(f"{'γ':>7s} {'night-mean N2O5 (ppt)':>22s} {'nitrate at 24 h (µg/m³)':>24s} {'% of ceiling':>13s}")
for g, r in sweep.items():
    nitrate = at.series(r, "NIT_ugm3")[-1]
    print(f"{g:7g} {1e3 * at.night_mean(r, 'N2O5'):22.0f} {nitrate:24.2f} {100 * nitrate / ceiling:13.0f}")

fig = go.Figure(go.Bar(x=gamma_labels, y=[at.series(r, "NIT_ugm3")[-1] for r in sweep.values()],
                       marker_color=config.SERIES_COLORS[0], name="model, after 24 h"))
fig.add_hline(y=ceiling, line=dict(color="#444444", dash="dash"),
              annotation_text=f"ceiling 2∫k₁[NO₂][O₃]dt ≈ {ceiling:.1f} µg m⁻³")
fig.update_yaxes(title="Nitrate (µg m⁻³)")
plots.style(fig, "Particle nitrate after 24 h")

## 6. Exercise 2: summer vs winter, and a high-latitude site

Only the sun changes between these runs: temperature, RH and the starting gases stay the same.
Each run starts at local noon. The cell runs Buffalo on 15 January and Hyytiälä (southern
Finland) on 1 July (2 runs, about 45 s) and compares them with the preset.

**Try this:** change `WINTER` or `HIGH_LATITUDE` to another site or date, for example
`("Fort Collins, CO", "2026-12-21")` or `("Manaus, Brazil", "2026-07-01")`.

In [ ]:
WINTER = ("Buffalo, NY", "2026-01-15")
HIGH_LATITUDE = ("Hyytiälä, Finland", "2026-07-01")

places = {"Buffalo, 2026-07-01": base}
for site, day in (WINTER, HIGH_LATITUDE):
    places[f"{site.split(',')[0]}, {day}"] = at.run(PRESET, name=f"{site}, {day}",
                                                    **at.site_overrides(site, day))

In [ ]:
plots.time_series(list(places.values()), ["J_NO2", "O3", "N2O5", "SOA_ugm3", "NIT_ugm3"],
                  labels=list(places)).show()

print(f"{'run':22s} {'night (h)':>9s} {'max J(NO₂)':>10s} {'O₃ night':>9s} {'N₂O₅ night':>10s} "
      f"{'ceiling':>8s} {'nitrate':>8s} {'SOA':>6s}")
print(f"{'':22s} {'':>9s} {'(s⁻¹)':>10s} {'(ppb)':>9s} {'(ppt)':>10s} {'(µg/m³)':>8s} {'(µg/m³)':>8s} {'':>6s}")
for label, r in places.items():
    night_h = sum(b - a for a, b in at.night_intervals(r))
    print(f"{label:22s} {night_h:9.1f} {at.series(r, 'J_NO2').max():10.2e} {at.night_mean(r, 'O3'):9.0f} "
          f"{1e3 * at.night_mean(r, 'N2O5'):10.0f} {nitrate_ceiling_ugm3(r):8.1f} "
          f"{at.series(r, 'NIT_ugm3')[-1]:8.2f} {at.series(r, 'SOA_ugm3')[-1]:6.1f}")

The shaded night bands belong to the first run (Buffalo in July); the other runs have their own
nights (see the table).

What to look for:

- **Hyytiälä in July:** the sun barely sets (SZA stays below about 95°). The short night leaves
  less time for N₂O₅ chemistry, and nitrate ends at about half the Buffalo value.
- **Buffalo in January:** weaker noon sun, and the longest night. O₃ grows less by day, so the
  night starts with less O₃. Yet the ceiling is no lower than in July, while N₂O₅ and nitrate
  are. Look at SOA during the January night: it keeps rising. What else is NO₃ reacting with?
- Temperature stays at 20 °C in every run. A real winter night would also shift
  NO₃ + NO₂ ⇌ N₂O₅ towards N₂O₅ (see "Going further").

## 7. Exercise 3: what does holding NO₂ constant mean?

The preset has `hold_constant = ("NO2",)`. After every 15-minute step the driver resets NO₂ to
its starting value of 1 ppb. That works like an emission source that replaces exactly what the
chemistry removed, at whatever rate is needed. It is a convenient stand-in for a city's
continuous NOx emissions, but the source is unlimited and has no daily cycle of its own. Here
are the lines of `driver.run` that do it:

In [ ]:
show_source_lines(driver.run, "held = ", "held_values")

Now run the same day with nothing held (`hold_constant=[]`, 1 run, about 25 s). A box has no
emissions, so the 1.5 ppb of NOx it starts with is all it gets.

In [ ]:
free = at.run(PRESET, hold_constant=[], name="NO₂ not held")
plots.time_series([base, free], ["NO2", "O3", "N2O5", "NIT_ugm3"],
                  labels=["NO₂ held at 1 ppb", "NO₂ not held"], log_y=True)

### Under the hood: a nitrogen budget

Count the nitrogen in NO, NO₂, HNO₃, N₂O₅ (two N each) and particle nitrate at the start and
at the end. Organic nitrates and PANs are not counted, and HNO₃ slowly leaves through the gas-phase
sink `<33>` (about 1 ppb of N over the day with NO₂ held; ASSUMPTIONS A2), so the end value is a
lower bound.

In [ ]:
def inorganic_nitrogen_ppb(result, i):
    """N (ppb) in NO, NO2, HNO3, N2O5 and particle nitrate at output i (organic nitrates not counted)."""
    gas = sum(at.series(result, k)[i] for k in ("NO", "NO2", "HNO3")) + 2 * at.series(result, "N2O5")[i]
    return gas + at.series(result, "NIT_ugm3")[i] / at.ugm3_per_ppb("NO3", result)


print(f"{'run':20s} {'N at start':>11s} {'N at 24 h':>10s} {'HNO₃ at 24 h':>13s} {'nitrate at 24 h':>16s}")
for label, r in (("NO₂ held at 1 ppb", base), ("NO₂ not held", free)):
    print(f"{label:20s} {inorganic_nitrogen_ppb(r, 0):9.2f}   {inorganic_nitrogen_ppb(r, -1):8.2f}   "
          f"{at.series(r, 'HNO3')[-1]:9.2f} ppb {at.series(r, 'NIT_ugm3')[-1]:10.2f} µg/m³")

With NO₂ held, the box ends the day with several times more inorganic nitrogen than it started
with: the extra came from the hold, acting as a NOx source. Without the hold, NO₂ falls to about
0.1 ppb within a few hours. Little NO₃ and N₂O₅ form at night, and particle nitrate stays near
zero. The rest of the starting NOx ends up in HNO₃ and in organic nitrates and PANs.

**Try this:** raise the held NO₂ with `at.run(PRESET, gases_ppb={"NO2": 20})`. The `gases_ppb`
you pass is merged into the preset's gases, so the other gases keep their values. What happens to
N₂O₅ and nitrate at night, and is the result realistic for a single air mass?

## 8. Your own runs: a run panel

Pick γ, a site, a start date and a local start hour, then press **Run 24 h**. Each run takes about
20–30 s. The plots compare the new run with the one before it (the first comparison is with the
preset run from section 2). The panel passes a function to `at.run(progress=...)` that moves its
progress bar.

In [ ]:
gamma_box = widgets.SelectionSlider(options=[0.0, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2],
                                    value=0.02, description="γ(N₂O₅)", continuous_update=False)
site_box = widgets.Dropdown(options=list(config.SITES), value="Buffalo, NY", description="Site")
date_box = widgets.DatePicker(value=date(2026, 7, 1), description="Start date")
hour_box = widgets.IntSlider(value=12, min=0, max=23, description="Local hour")
hold_box = widgets.Checkbox(value=True, description="Hold NO₂ at its initial value")
run_button = widgets.Button(description="Run 24 h", button_style="primary", icon="play")
progress_bar = widgets.FloatProgress(value=0.0, min=0.0, max=1.0, layout=widgets.Layout(width="240px"))
status = widgets.HTML()
panel_out = widgets.Output()
history = [base]


def run_panel(_=None):
    if date_box.value is None:
        status.value = "Pick a start date first."
        return
    label = (f"{site_box.value}, {date_box.value:%d %b}, {hour_box.value:02d}:00, γ = {gamma_box.value:g}"
             + ("" if hold_box.value else ", NO₂ not held"))
    run_button.disabled = True
    progress_bar.value = 0.0
    status.value = f"Running {html.escape(label)} ..."
    try:
        result = at.run(PRESET, name=label, gamma_n2o5=gamma_box.value,
                        hold_constant=["NO2"] if hold_box.value else [],
                        progress=lambda fraction: setattr(progress_bar, "value", fraction),
                        **at.site_overrides(site_box.value, date_box.value, hour_box.value))
    except Exception as err:   # show the error here; widget callbacks otherwise hide it
        status.value = f"<b>Run failed:</b> {html.escape(str(err))}"
        return
    finally:
        run_button.disabled = False
    previous = history[-1]
    history.append(result)
    warnings = "".join(f"<br>Warning: {html.escape(w)}" for w in result.diagnostics.get("warnings", []))
    status.value = (f"Done in {result.diagnostics['runtime_s']:.0f} s. Nitrate after 24 h: "
                    f"<b>{at.series(result, 'NIT_ugm3')[-1]:.2f} µg/m³</b> "
                    f"(previous run: {at.series(previous, 'NIT_ugm3')[-1]:.2f}).{warnings}")
    with panel_out:
        panel_out.clear_output(wait=True)
        plots.time_series([result, previous], ["J_NO2", "NO3", "N2O5", "NIT_ugm3"],
                          labels=["this run", "previous run"]).show()
        plots.composition(result).show()


run_button.on_click(run_panel)
display(widgets.VBox([widgets.HBox([gamma_box, site_box]), widgets.HBox([date_box, hour_box]),
                      hold_box, widgets.HBox([run_button, progress_bar]), status, panel_out]))

Every run is kept in the list `history`, so you can plot them together afterwards, for example
`plots.time_series(history, ["N2O5", "NIT_ugm3"])` or `plots.compare_size_distributions(history[-2:])`.

## Questions to think about

1. N₂O₅ stays below 1 ppt all afternoon, although NO₂ and O₃ are present. Which two daytime
   reactions keep NO₃, and with it N₂O₅, so low?
2. During the night nitrate grows almost linearly in time. What does that tell you about N₂O₅
   and $k_{het}$ during the night?
3. OH is higher on the second morning than on the first afternoon. Compare limonene and O₃ at the
   two times: what changed in the OH sources and sinks?
4. In the radicals plot, when does NO₃ take over from OH as the main oxidant? How large is the
   night-time NO₃ concentration compared with the daytime OH concentration?
5. In the γ sweep, a factor of 5 in γ (0.02 → 0.1) gives much less than a factor of 5 in
   nitrate. Use the ceiling argument to explain why. When would γ matter most?
6. Real γ falls as particles gain nitrate and organic coatings. How would that change the
   γ = 0.02 result over the night?
7. Holding NO₂ constant stands in for emissions. What does it leave out (daily emission cycle,
   boundary-layer height, dilution, deposition), and which results above would change most?

## Going further

- **Cold winter night:** `at.run(PRESET, environment={"temp_k": 268.15}, **at.site_overrides("Buffalo, NY", "2026-01-15"))`.
  Colder air shifts NO₃ + NO₂ ⇌ N₂O₅ towards N₂O₅. It also lowers the c* of the organics, so SOA
  changes as well.
- **Humidity:** `environment={"rh": 0.9}` gives more particle water and a larger wet surface, so
  $k_{het}$ rises (by more than it would for real SOA, which takes up less water than the model
  assumes; A9). `processes={"n2o5_uptake": False}` switches the uptake off altogether.
- **Two days:** `duration_h=48` (any multiple of 0.25 h works) shows whether the second night
  repeats the first.
- **More limonene:** `gases_ppb={"LIMONENE": 10}`. SOA rises to about 70 µg m⁻³, more than the mass
  of the limonene itself (10 ppb ≈ 57 µg m⁻³). The NO₂ hold supplies unlimited nitrogen for the PANs
  and organic nitrates that the placeholder c\* makes condensable (ASSUMPTIONS A8, A14), so treat
  this as an artefact. Nitrate barely changes (5.1 → 5.7 µg m⁻³), because NO₃ now also reacts with
  limonene and its products and night-time N₂O₅ falls by about 60 %.
- **Raw API:** `from tomas_jax.box import scenario, driver`, then
  `s = scenario.load_preset(PRESET).replace(gamma_n2o5=0.1, numerics={"nbins": 80})` and `driver.run(s)`.
  This is what `at.run` does, plus a progress bar and a timeout (the preset itself has 40 size bins,
  like the web app).
- **Read the assumptions:** [ASSUMPTIONS.md](https://github.com/reflective-org/TOMAS-JAX/blob/1f519e910b5e3119a54ce3df1617aa32a4d5ab2c/ASSUMPTIONS.md) (A1, A2, A5, A7, A8, A9, A11,
  A12 and A14 matter most here) and [docs/aaar/heterogeneous.md](https://github.com/reflective-org/TOMAS-JAX/blob/1f519e910b5e3119a54ce3df1617aa32a4d5ab2c/docs/aaar/heterogeneous.md).

### References

- Bertram, T. H. and Thornton, J. A.: Toward a general parameterization of N₂O₅ reactivity on
  aqueous particles, *Atmos. Chem. Phys.*, 9, 8351–8363, 2009.
- Brown, S. S. and Stutz, J.: Nighttime radical observations and chemistry, *Chem. Soc. Rev.*,
  41, 6405–6447, 2012.
- Burkholder, J. B., et al.: Chemical Kinetics and Photochemical Data for Use in Atmospheric
  Studies, Evaluation No. 19, JPL Publication 19-5, 2019.
- Chang, W. L., et al.: Heterogeneous atmospheric chemistry, ambient measurements, and model
  calculations of N₂O₅: a review, *Aerosol Sci. Technol.*, 45, 665–695, 2011.
- Li, Y., Pöschl, U. and Shiraiwa, M.: Molecular corridors and parameterizations of volatility in
  the chemical evolution of organic aerosols, *Atmos. Chem. Phys.*, 16, 3327–3344, 2016.
- Madronich, S. and Flocke, S.: The role of solar radiation in atmospheric chemistry, in
  *Handbook of Environmental Chemistry*, Springer, 1–26, 1999 (TUV).
- Seinfeld, J. H. and Pandis, S. N.: *Atmospheric Chemistry and Physics*, 3rd ed., Wiley, 2016.